# Data Mining Assignment – Spark Data Lake Preprocessing
**Dataset:** `sales.csv`  |  **Environment:** Google Colab + PySpark

Bronze = raw data (unchanged). Silver = basic preprocessed data. No Gold layer.

## 1. Install PySpark & create the Data Lake folders

In [20]:
!pip install -q pyspark

In [21]:
import os, shutil, glob

BASE   = "data_lake"
BRONZE = f"{BASE}/bronze"
SILVER = f"{BASE}/silver"
SILVER_OUT = f"{SILVER}/preprocessed_sales"

os.makedirs(BRONZE, exist_ok=True)
os.makedirs(SILVER_OUT, exist_ok=True)
print("Folders created:", BRONZE, SILVER_OUT)

Folders created: data_lake/bronze data_lake/silver/preprocessed_sales


### Upload `sales.csv` (Colab) and copy it *unchanged* into Bronze

In [3]:
# In Colab a file picker opens. If sales.csv is already in the working folder, the upload is skipped.
if not os.path.exists("sales.csv"):
    from google.colab import files
    files.upload()          # choose sales.csv

shutil.copy("sales.csv", f"{BRONZE}/sales.csv")   # raw copy – never modified
print(os.listdir(BRONZE))

Saving sales.csv to sales.csv
['sales.csv']


## 2. Load and inspect with Spark

In [4]:
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder.appName("SalesDataLake").master("local[*]")
         .config("spark.sql.ansi.enabled", "false")      # invalid dates -> null instead of error
         .getOrCreate())

# Read everything as STRING so bad values (e.g. "not-a-date") are not hidden by schema inference
raw = spark.read.csv(f"{BRONZE}/sales.csv", header=True, inferSchema=False)
raw.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [5]:
raw.show(5, truncate=False)
raw_count = raw.count()
print("Total records (Bronze):", raw_count)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra|Saree        |Clothing|2       |6710      |20              |Credit Card   |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh   |T-Shirt      |Clothing|6       |680       |5               |Debit Card    |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao   |Hoodie       |Clothing|6       |3080      |12              |UPI           |Pune      |Maharashtra   |2025-05-06|Shipped     |
|100004  |C0051      |

### 2.1 Duplicates

In [6]:
distinct_count = raw.dropDuplicates().count()
print("Exact duplicate rows:", raw_count - distinct_count)

dup_ids = raw.groupBy("order_id").count().filter("count > 1").count()
print("order_id values appearing more than once:", dup_ids)

Exact duplicate rows: 10
order_id values appearing more than once: 18


### 2.2 Null / blank values

In [7]:
def blank(c):
    return F.col(c).isNull() | (F.trim(F.col(c)) == "")

raw.select([F.sum(blank(c).cast("int")).alias(c) for c in raw.columns]).show(vertical=True)

-RECORD 0---------------
 order_id         | 0   
 customer_id      | 0   
 customer_name    | 10  
 product          | 0   
 category         | 0   
 quantity         | 0   
 unit_price       | 0   
 discount_percent | 0   
 payment_method   | 8   
 city             | 8   
 state            | 0   
 order_date       | 0   
 order_status     | 0   



### 2.3 Text-format inconsistencies (extra spaces, mixed case)

In [8]:
text_cols = ["customer_name","product","category","payment_method","city","state","order_status"]

# extra / leading / trailing spaces
raw.select([F.sum((F.col(c) != F.trim(F.regexp_replace(F.col(c), r"\s+", " "))).cast("int")).alias(c)
            for c in text_cols]).show(vertical=True)

# case variants: distinct raw values vs distinct values after lower+trim
for c in ["category","city"]:
    d1 = raw.select(c).distinct().count()
    d2 = raw.select(F.lower(F.trim(F.col(c)))).distinct().count()
    print(f"{c}: {d1} distinct raw values -> {d2} after lower/trim")
raw.groupBy("category").count().orderBy("category").show(20, False)

-RECORD 0-------------
 customer_name  | 3   
 product        | 0   
 category       | 0   
 payment_method | 0   
 city           | 3   
 state          | 0   
 order_status   | 0   

category: 13 distinct raw values -> 8 after lower/trim
city: 19 distinct raw values -> 12 after lower/trim
+---------------+-----+
|category       |count|
+---------------+-----+
|Beauty         |136  |
|Books          |128  |
|CLOTHING       |1    |
|Clothing       |108  |
|ELECTRONICS    |1    |
|Electronics    |103  |
|Furniture      |132  |
|Grocery        |136  |
|Home Appliances|115  |
|SPORTS         |1    |
|Sports         |136  |
|beauty         |2    |
|sports         |1    |
+---------------+-----+



### 2.4 Invalid numerical values

In [9]:
num = (raw.withColumn("q", F.col("quantity").cast("int"))
          .withColumn("p", F.col("unit_price").cast("double"))
          .withColumn("d", F.col("discount_percent").cast("double")))

num.select(
    F.sum((F.col("q").isNull() & ~blank("quantity")).cast("int")).alias("non_numeric_quantity"),
    F.sum((F.col("q") <= 0).cast("int")).alias("quantity_le_0"),
    F.sum((F.col("p") <= 0).cast("int")).alias("unit_price_le_0"),
    F.sum(((F.col("d") < 0) | (F.col("d") > 100)).cast("int")).alias("discount_outside_0_100"),
).show()
num.select("q","p","d").describe().show()

+--------------------+-------------+---------------+----------------------+
|non_numeric_quantity|quantity_le_0|unit_price_le_0|discount_outside_0_100|
+--------------------+-------------+---------------+----------------------+
|                   0|           12|              8|                     8|
+--------------------+-------------+---------------+----------------------+

+-------+------------------+------------------+------------------+
|summary|                 q|                 p|                 d|
+-------+------------------+------------------+------------------+
|  count|              1000|              1000|              1000|
|   mean|             5.355|            5702.4|            19.948|
| stddev|2.9916425128890194|10990.194484849431|14.236782711763993|
|    min|                -2|            -500.0|              -5.0|
|    max|                10|           81010.0|             120.0|
+-------+------------------+------------------+------------------+



### 2.5 Inconsistent / invalid dates

In [10]:
FORMATS = {"yyyy-MM-dd": r"^\d{4}-\d{2}-\d{2}$",
           "yyyy/MM/dd": r"^\d{4}/\d{2}/\d{2}$",
           "dd/MM/yyyy": r"^\d{2}/\d{2}/\d{4}$",
           "MM-dd-yyyy": r"^\d{2}-\d{2}-\d{4}$"}

for fmt, rx in FORMATS.items():
    print(f"{fmt:12s}:", raw.filter(F.col("order_date").rlike(rx)).count())

known = "|".join(f"({rx})" for rx in FORMATS.values())
print("Unrecognised pattern:", raw.filter(~F.col("order_date").rlike(known)).count())
# values that look like a date but are impossible (e.g. 2025-02-30, 2026-13-05)
impossible = raw.filter(F.col("order_date").rlike(FORMATS["yyyy-MM-dd"]) & F.to_date("order_date", "yyyy-MM-dd").isNull())
print("Well-formatted but impossible dates:", impossible.count())
impossible.select("order_id","order_date").show()

print("\nNon-standard (not yyyy-MM-dd) date values:")
raw.filter(~F.col("order_date").rlike(FORMATS["yyyy-MM-dd"])).select("order_id","order_date").show(truncate=False)

yyyy-MM-dd  : 996
yyyy/MM/dd  : 1
dd/MM/yyyy  : 1
MM-dd-yyyy  : 1
Unrecognised pattern: 1
Well-formatted but impossible dates: 2
+--------+----------+
|order_id|order_date|
+--------+----------+
|  100634|2025-02-30|
|  100637|2026-13-05|
+--------+----------+


Non-standard (not yyyy-MM-dd) date values:
+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100358  |2025/07/14|
|100448  |not-a-date|
|100969  |09-15-2026|
+--------+----------+



## 3. Preprocessing → Silver layer

**Step 1 – Remove exact duplicate rows**

In [11]:
df = raw.dropDuplicates()
print("After duplicate removal:", df.count())

After duplicate removal: 990


**Step 2 – Trim spaces & standardise capitalisation** (done *before* missing-value handling so whitespace-only values count as missing)

In [12]:
for c in df.columns:
    df = df.withColumn(c, F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))
    df = df.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))   # blank -> null

# Title-case only fields where case varies (product names like "USB-C Hub" are left untouched)
for c in ["category", "city", "customer_name"]:
    df = df.withColumn(c, F.initcap(F.col(c)))
df.groupBy("category").count().show()
df.groupBy("city").count().orderBy("city").show(30)

+---------------+-----+
|       category|count|
+---------------+-----+
|         Sports|  138|
|        Grocery|  134|
|    Electronics|  104|
|       Clothing|  108|
|          Books|  127|
|Home Appliances|  113|
|      Furniture|  129|
|         Beauty|  137|
+---------------+-----+

+----------+-----+
|      city|count|
+----------+-----+
|      NULL|    8|
| Bengaluru|   90|
|   Chennai|   88|
|Coimbatore|   81|
|     Delhi|   83|
| Hyderabad|   97|
|    Jaipur|   83|
|   Kolkata|   81|
|   Lucknow|   89|
|    Mumbai|  105|
|      Pune|   81|
|Vijayawada|  104|
+----------+-----+



**Step 3 – Missing values:** text gaps are filled with Unknown so the record is kept.

In [13]:
df = df.fillna({"customer_name": "Unknown", "payment_method": "Unknown", "city": "Unknown"})

**Step 4 – Validate numeric columns**\n- `quantity` ≤ 0 or `unit_price` ≤ 0 → record is meaningless (no valid sale) → removed\n- `discount_percent` < 0 or > 100 → set to 0 (no discount) so the record is preserved

In [14]:
df = (df.withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn("unit_price", F.col("unit_price").cast("double"))
        .withColumn("discount_percent", F.col("discount_percent").cast("double")))

before = df.count()
df = df.filter((F.col("quantity") > 0) & (F.col("unit_price") > 0))
print("Rows removed for invalid quantity/unit_price:", before - df.count())

bad_disc = df.filter((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).count()
print("Invalid discounts reset to 0:", bad_disc)
df = df.withColumn("discount_percent",
        F.when((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100), 0.0)
         .otherwise(F.col("discount_percent")))

Rows removed for invalid quantity/unit_price: 20
Invalid discounts reset to 0: 8


**Step 5 – Standardise dates to `yyyy-MM-dd`**; impossible dates (e.g. 31/02/2026, `not-a-date`) become null but the record is kept.

In [15]:
parsed = F.coalesce(*[F.when(F.col("order_date").rlike(rx), F.to_date(F.col("order_date"), fmt))
                      for fmt, rx in FORMATS.items()])
df = df.withColumn("order_date", F.date_format(parsed, "yyyy-MM-dd"))
print("Invalid dates set to null:", df.filter(F.col("order_date").isNull()).count())

Invalid dates set to null: 4


**Step 6 – Final casting & save to Silver** (Parquet + CSV)

In [16]:
silver = df.withColumn("order_date", F.to_date("order_date")).orderBy("order_id")

# Spark output folder (Parquet)
silver.write.mode("overwrite").parquet(f"{SILVER_OUT}/parquet")

# Spark output folder (CSV) + single file required by the submission instructions
silver.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{SILVER_OUT}/csv")
part = glob.glob(f"{SILVER_OUT}/csv/part-*.csv")[0]
shutil.copy(part, f"{SILVER}/sales_silver.csv")
print(os.listdir(SILVER), os.listdir(SILVER_OUT))

['sales_silver.csv', 'preprocessed_sales'] ['parquet', 'csv']


## 4. Verification

In [17]:
silver_count = silver.count()
print(f"Rows BEFORE preprocessing (Bronze): {raw_count}")
print(f"Rows AFTER  preprocessing (Silver): {silver_count}")
print(f"Rows removed: {raw_count - silver_count}")

Rows BEFORE preprocessing (Bronze): 1000
Rows AFTER  preprocessing (Silver): 970
Rows removed: 30


In [18]:
check = spark.read.parquet(f"{SILVER_OUT}/parquet")
check.printSchema()
check.show(10, truncate=False)
print("Silver rows read back:", check.count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+-------------

In [19]:
# Bronze must be untouched: compare with the original upload
import filecmp
print("Bronze identical to original:", filecmp.cmp("sales.csv", f"{BRONZE}/sales.csv", shallow=False))

Bronze identical to original: True


### Preprocessing summary
1. **Duplicates** – exact duplicate rows removed with `dropDuplicates()`.
2. **Text cleaning** – extra/leading/trailing spaces trimmed; capitalisation standardised for `category`, `city`, `customer_name`.
3. **Missing values** – blank `customer_name`, `payment_method`, `city` filled with `"Unknown"`.
4. **Numeric validation** – rows with `quantity ≤ 0` or `unit_price ≤ 0` removed; `discount_percent` outside 0–100 reset to 0.
5. **Dates** – four mixed formats converted to `yyyy-MM-dd`; impossible/invalid dates set to null (record kept).
6. **Output** – saved to `data_lake/silver/preprocessed_sales/` (Parquet + CSV) and `data_lake/silver/sales_silver.csv`.